In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from collections import defaultdict, deque
from multiprocessing import Manager
from scipy.optimize import linear_sum_assignment
from tqdm.contrib.concurrent import process_map
import networkx as nx
import random
import heapq
import json
from copy import deepcopy
from utils import lineage_name_mapping, load_json, bidict
from zss import simple_distance, Node
from matplotlib import pyplot as plt
from typing import List, Tuple, Dict, Any
from functools import partial
from itertools import combinations

from pareto_core import LineageOptimization, LineageTree

In [ ]:
lineage_data = load_json('./data/cell_lineage.json')
lineage_exp_df = pd.read_csv("data/protein/aggregated_all/s3.csv", index_col=0)
lineage_exp_df = lineage_exp_df.fillna(0)
tracking_df_all = pd.read_csv("./data/embryo2/tracks.txt", sep="\t")
tracking_time_cutoff = 242
tracking_scale = 0.1625
tracking_df = tracking_df_all.loc[tracking_df_all["t"] <= tracking_time_cutoff]
untracked_nodes = ["P0", "AB", "P1"]
first_internal_layer = ["ABa", "ABp", "EMS", "P2"]

In [ ]:
cell_names = tracking_df["name"].unique()
lineage_xyz_df = pd.DataFrame(columns=["x", "y", "z"])
for name in cell_names:
    time_points = tracking_df.loc[tracking_df['name'] == name]["t"].values
    if len(time_points) == 1 and time_points[0] == tracking_time_cutoff:
        continue
    last_xyz_coordinates = tracking_df.loc[tracking_df['name'] == name][["x", "y", "z"]].values[-1]
    # appending last_xyz_coordinates to lineage_xyz_df as a new row with cell name as index
    lineage_xyz_df.loc[name] = last_xyz_coordinates * tracking_scale

In [ ]:
common_lineages = lineage_xyz_df.index.intersection(lineage_exp_df.index)
for name in untracked_nodes:
    lineage_xyz_df.loc[name] = np.repeat(np.nan, 3)
    lineage_exp_df.loc[name] = np.repeat(np.nan, lineage_exp_df.shape[1])
common_lineages = untracked_nodes + common_lineages.tolist()

In [ ]:
lineage_tree = LineageTree()
# add first three nodes: ["AB", "P0", "P1"]
lineage_tree.add_node(0, -1)
lineage_tree.root = 0
lineage_tree.add_node(1, 0)
lineage_tree.add_node(2, 0)
bfs_queue = deque()
bfs_queue.append((lineage_data, -1))
while bfs_queue:
    node, parent_idx = bfs_queue.popleft()
    curr_name = lineage_name_mapping(node['did'])
    if curr_name not in common_lineages:
        continue
    curr_idx = common_lineages.index(curr_name)
    if curr_idx >= 3:
        parent_idx = lineage_tree.reverse_lineage_id_mapping[parent_idx]
        lineage_tree.add_node(curr_idx, parent_idx)
    children = node.get('children', [])
    for child_node in children:
        bfs_queue.append((child_node, curr_idx))

print("first layer of tracked nodes:", lineage_tree.children_list[1]+lineage_tree.children_list[2])

In [ ]:
lineage_xyz_mat = lineage_xyz_df.loc[common_lineages].values
lineage_exp_mat = lineage_exp_df.loc[common_lineages].values

In [ ]:
first_layer = [(3,2), (4,2), (5,2), (6,2)]

opt = LineageOptimization(
    lineage_xyz_mat, 
    lineage_exp_mat, 
    lineage_tree,
    first_internal_layer=first_layer,
    lineage_names=common_lineages)

In [ ]:
test_mst = opt.mst_test(1000)

In [ ]:
mst_list = opt.mst_test_runner()

In [ ]:
test_mst.size(weight='weight')

In [ ]:
len(opt.terminal_tree_ids)

In [ ]:
max_degree = 0
exceeding_edges = 0
exceeding_nodes = 0
for i, j in mst.adjacency():
    max_degree = max(max_degree, len(j))
    if len(j) > 3:
        exceeding_edges += len(j) - 3
        exceeding_nodes += 1
print("Max degree:", max_degree)
print("Exceeding edges:", exceeding_edges)
print("Exceeding nodes:", exceeding_nodes)

In [ ]:
pareto_list_by_layer = opt.bottom_up_by_layer_runner()
pareto_list_top_down_rebuild = opt.top_down_rebuild_runner()

In [ ]:
pareto_list_paired_bottom_up_rebuild = opt.paired_bottom_up_rebuild_runner()

In [ ]:
pareto_list_top_down_balanced_rebuild = opt.top_down_balanced_rebuild_runner()

In [ ]:
pareto_list_direct_bottom_up_rebuild = opt.direct_bottom_up_rebuild_runner()

In [ ]:
alpha_tree_tops_list = []
for i in range(1001):
    alpha = 0 + 0.001 * i
    alpha_tree_tops_list.append((alpha, pareto_list_direct_bottom_up_rebuild[i][4]))
# plot alpha vs tree tops
alphas = [x[0] for x in alpha_tree_tops_list]
tree_tops = [x[1] for x in alpha_tree_tops_list]
plt.figure(figsize=(6,4))
plt.plot(alphas, tree_tops, marker='o', markersize=2)
plt.xlabel('Alpha (weight for XYZ cost)')
plt.ylabel('Number of tree tops after direct bottom-up rebuild')
plt.title('Alpha vs Number of Tree Tops')
plt.grid(True)

In [ ]:
pareto_xyz_cost_list_by_layer = [cost[0] for cost in pareto_list_by_layer]
pareto_exp_cost_list_by_layer = [cost[1] for cost in pareto_list_by_layer]
pareto_xyz_cost_list_top_down_rebuild = [cost[0] for cost in pareto_list_top_down_rebuild]
pareto_exp_cost_list_top_down_rebuild = [cost[1] for cost in pareto_list_top_down_rebuild]
pareto_xyz_cost_list_direct_bottom_up_rebuild = [cost[0] for cost in pareto_list_direct_bottom_up_rebuild]
pareto_exp_cost_list_direct_bottom_up_rebuild = [cost[1] for cost in pareto_list_direct_bottom_up_rebuild]
pareto_xyz_cost_list_paired_bottom_up_rebuild = [cost[0] for cost in pareto_list_paired_bottom_up_rebuild]
pareto_exp_cost_list_paired_bottom_up_rebuild = [cost[1] for cost in pareto_list_paired_bottom_up_rebuild]
plt.figure(figsize=(8, 6))
plt.plot(pareto_xyz_cost_list_by_layer, pareto_exp_cost_list_by_layer, marker='o', linestyle='-',  markersize=3, label='Bottom-up By Layer')
plt.plot(pareto_xyz_cost_list_top_down_rebuild, pareto_exp_cost_list_top_down_rebuild, marker='o', linestyle='-',  markersize=3, label='Top-Down Rebuild')
plt.plot(pareto_xyz_cost_list_direct_bottom_up_rebuild, pareto_exp_cost_list_direct_bottom_up_rebuild, marker='o', linestyle='-', markersize=3, label='Direct Bottom-Up Rebuild')
plt.plot(pareto_xyz_cost_list_paired_bottom_up_rebuild, pareto_exp_cost_list_paired_bottom_up_rebuild, marker='o', linestyle='-', markersize=3, label='Paired Bottom-Up Rebuild')
plt.xlabel('Motility Cost')
plt.ylabel('Expression Cost')
plt.scatter(opt.lineage_xyz_cost, opt.lineage_exp_cost, color='black', marker='*', label='Lineage Cost', zorder=99)
plt.title(f'Full Tree Pareto Front, t=242, avg exp')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# save results to csv
import pandas as pd
results_df = pd.DataFrame({
    'Bottom-up By Layer XYZ Cost': pareto_xyz_cost_list_by_layer,
    'Bottom-up By Layer Exp Cost': pareto_exp_cost_list_by_layer,
    'Top-Down Rebuild XYZ Cost': pareto_xyz_cost_list_top_down_rebuild,
    'Top-Down Rebuild Exp Cost': pareto_exp_cost_list_top_down_rebuild,
    'Top-Down Balanced Rebuild XYZ Cost': pareto_xyz_cost_list_top_down_balanced_rebuild,
    'Top-Down Balanced Rebuild Exp Cost': pareto_exp_cost_list_top_down_balanced_rebuild,
    'Paired Bottom-Up Rebuild XYZ Cost': pareto_xyz_cost_list_paired_bottom_up_rebuild,
    'Paired Bottom-Up Rebuild Exp Cost': pareto_exp_cost_list_paired_bottom_up_rebuild
})
results_df.to_csv('output/internal_opt/s3_242_3d_l2/pareto_front.csv', index=False)

In [ ]:
pareto_list_direct_bottom_up_rebuild = opt.direct_bottom_up_rebuild_runner()

In [ ]:
pareto_list_terminal_only = opt.terminal_only_rebuild_runner()

In [ ]:
pareto_list_paired_bottom_up_rebuild = opt.paired_bottom_up_rebuild_runner()

In [ ]:
# all tree bottom up by layer pareto optimization
pareto_list_by_layer = opt.bottom_up_by_layer_runner()
pareto_list_by_cell = opt.bottom_up_by_cell_runner()


In [ ]:
pareto_xyz_cost_list_by_layer = [cost[0] for cost in pareto_list_by_layer]
pareto_exp_cost_list_by_layer = [cost[1] for cost in pareto_list_by_layer]
pareto_xyz_cost_list_by_cell = [cost[0] for cost in pareto_list_by_cell]
pareto_exp_cost_list_by_cell = [cost[1] for cost in pareto_list_by_cell]
plt.figure(figsize=(8, 6))
plt.plot(pareto_xyz_cost_list_by_layer, pareto_exp_cost_list_by_layer, marker='o', linestyle='-', color='blue', markersize=3, label='Pareto Front (By Layer)')
plt.plot(pareto_xyz_cost_list_by_cell, pareto_exp_cost_list_by_cell, marker='o', linestyle='-', color='green', markersize=3, label='Pareto Front (By Cell)')
plt.xlabel('Motility Cost')
plt.ylabel('Expression Cost')
plt.scatter(opt.lineage_xyz_cost, opt.lineage_exp_cost, color='red', label='Lineage Cost')
plt.title(f'Full tree bottom up, t=248, avg exp')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
first_layer_names = ['ABa', 'ABp', 'P2', 'EMS']
pareto_list_by_layer_subtree = []
pareto_list_by_cell_subtree = []
lineage_cost_subtree = []
for name, layer in zip(first_layer_names, first_layer):
    subtree_pareto_list_by_layer = opt.bottom_up_by_layer_runner(first_internal_layer=[layer])
    subtree_pareto_list_by_cell = opt.bottom_up_by_cell_runner(first_internal_layer=[layer])
    pareto_list_by_layer_subtree.append(subtree_pareto_list_by_layer)
    pareto_list_by_cell_subtree.append(subtree_pareto_list_by_cell)
    lineage_cost_subtree.append(opt.calc_lineage_cost(first_internal_tree_ids=[layer[0]]))

In [ ]:
subtree_edge_counts = [360, 368, 184, 96]

In [ ]:
for edge_count, name, subtree_pareto_list_by_layer, subtree_pareto_list_by_cell, lineage_cost in \
        zip(subtree_edge_counts, first_layer_names, pareto_list_by_layer_subtree, pareto_list_by_cell_subtree, lineage_cost_subtree):
    pareto_xyz_cost_list_by_layer = [cost[0] for cost in subtree_pareto_list_by_layer]
    pareto_exp_cost_list_by_layer = [cost[1] for cost in subtree_pareto_list_by_layer]
    pareto_xyz_cost_list_by_cell = [cost[0] for cost in subtree_pareto_list_by_cell]
    pareto_exp_cost_list_by_cell = [cost[1] for cost in subtree_pareto_list_by_cell]
    plt.figure(figsize=(8, 6))
    plt.plot(pareto_xyz_cost_list_by_layer, pareto_exp_cost_list_by_layer, marker='o', linestyle='-', color='blue', markersize=3, label='Pareto Front (By Layer)')
    plt.plot(pareto_xyz_cost_list_by_cell, pareto_exp_cost_list_by_cell, marker='o', linestyle='-', color='green', markersize=3, label='Pareto Front (By Cell)')
    plt.xlabel('Motility Cost')
    plt.ylabel('Expression Cost')
    plt.scatter(lineage_cost[0], lineage_cost[1], color='red', label='Lineage Cost')
    plt.title(f'Subtree {name} bottom up, t=248, avg exp, edges={edge_count}')
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
pareto_list_by_layer_subtree_sum = np.array(pareto_list_by_layer_subtree).sum(axis=0)
pareto_list_by_cell_subtree_sum = np.array(pareto_list_by_cell_subtree).sum(axis=0)

pareto_xyz_cost_list_by_layer_subtree_aggregate = [cost[0] for cost in pareto_list_by_layer_subtree_sum]
pareto_exp_cost_list_by_layer_subtree_aggregate = [cost[1] for cost in pareto_list_by_layer_subtree_sum]
pareto_xyz_cost_list_by_cell_subtree_aggregate = [cost[0] for cost in pareto_list_by_cell_subtree_sum]
pareto_exp_cost_list_by_cell_subtree_aggregate = [cost[1] for cost in pareto_list_by_cell_subtree_sum]
pareto_xyz_cost_list_by_layer = [cost[0] for cost in pareto_list_by_layer]
pareto_exp_cost_list_by_layer = [cost[1] for cost in pareto_list_by_layer]
pareto_xyz_cost_list_by_cell = [cost[0] for cost in pareto_list_by_cell]
pareto_exp_cost_list_by_cell = [cost[1] for cost in pareto_list_by_cell]
plt.figure(figsize=(8, 6))
plt.plot(pareto_xyz_cost_list_by_layer, pareto_exp_cost_list_by_layer, marker='o', linestyle='-', color='blue', markersize=3, label='Pareto Front (By Layer)')
plt.plot(pareto_xyz_cost_list_by_cell, pareto_exp_cost_list_by_cell, marker='o', linestyle='-', color='green', markersize=3, label='Pareto Front (By Cell)')
plt.plot(pareto_xyz_cost_list_by_layer_subtree_aggregate, pareto_exp_cost_list_by_layer_subtree_aggregate, marker='o', linestyle='-', color='darkblue', markersize=3, label='Subtree combine by layer)')
plt.plot(pareto_xyz_cost_list_by_cell_subtree_aggregate, pareto_exp_cost_list_by_cell_subtree_aggregate, marker='o', linestyle='-', color='darkgreen', markersize=3, label='Subtree combine by cell)')
plt.xlabel('Motility Cost')
plt.ylabel('Expression Cost')
plt.scatter(opt.lineage_xyz_cost, opt.lineage_exp_cost, color='red', label='Lineage Cost', zorder=5)
plt.title(f'Bottom up full tree vs subtree combine, t=248, avg exp')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
pareto_list_by_layer